In [17]:
import pandas as pd

df = pd.read_csv("../data/Fashion_Retail_Sales.csv")

# PROFILING d'abord (le réflexe)
print("Dimensions :", df.shape)
print("\nColonnes :", df.columns.tolist())
print("\nValeurs manquantes :\n", df.isnull().sum())
df.head()

Dimensions : (3400, 6)

Colonnes : ['Customer Reference ID', 'Item Purchased', 'Purchase Amount (USD)', 'Date Purchase', 'Review Rating', 'Payment Method']

Valeurs manquantes :
 Customer Reference ID      0
Item Purchased             0
Purchase Amount (USD)    650
Date Purchase              0
Review Rating            324
Payment Method             0
dtype: int64


,Customer Reference ID,Item Purchased,Purchase Amount (USD),Date Purchase,Review Rating,Payment Method
0,4018,Handbag,4619.0,05-02-2023,NaN,Credit Card
1,4115,Tunic,2456.0,11-07-2023,2.0,Credit Card
2,4019,Tank Top,2102.0,23-03-2023,4.1,Cash
3,4097,Leggings,3126.0,15-03-2023,3.2,Cash
4,3997,Wallet,3003.0,27-11-2022,4.7,Cash


In [18]:
df_clean = df.copy()

# Renommer proprement (noms courts, sans espaces)
df_clean.columns = ["client_id", "article", "montant", "date", "note", "paiement"]

# Nettoyage
df_clean = df_clean[df_clean["montant"] > 0]              # montants valides
df_clean["date"] = pd.to_datetime(df_clean["date"], format="%d-%m-%Y")

# Dimensions temporelles (pour l'analyse de saisonnalité)
df_clean["mois"] = df_clean["date"].dt.to_period("M").astype(str)
df_clean["annee"] = df_clean["date"].dt.year
df_clean["jour_semaine"] = df_clean["date"].dt.day_name()

print("Avant :", len(df), "→ Après :", len(df_clean))
df_clean.head()

Avant : 3400 → Après : 2750


,client_id,article,montant,date,note,paiement,mois,annee,jour_semaine
0,4018,Handbag,4619.0,2023-02-05,NaN,Credit Card,2023-02,2023,Sunday
1,4115,Tunic,2456.0,2023-07-11,2.0,Credit Card,2023-07,2023,Tuesday
2,4019,Tank Top,2102.0,2023-03-23,4.1,Cash,2023-03,2023,Thursday
3,4097,Leggings,3126.0,2023-03-15,3.2,Cash,2023-03,2023,Wednesday
4,3997,Wallet,3003.0,2022-11-27,4.7,Cash,2022-11,2022,Sunday


In [19]:
# Agréger par client : combien d'achats, combien dépensé
clients = df_clean.groupby("client_id").agg(
    nb_achats=("article", "count"),
    total_depense=("montant", "sum")
).reset_index()

# Segmenter les clients par valeur (quartiles de dépense totale)
clients["segment_valeur"] = pd.qcut(
    clients["total_depense"], 4,
    labels=["1. Occasionnel", "2. Régulier", "3. Fidèle", "4. VIP"]
)

# Ramener le segment dans les transactions
df_final = df_clean.merge(
    clients[["client_id", "nb_achats", "total_depense", "segment_valeur"]],
    on="client_id", how="left"
)

df_final.head()

,client_id,article,montant,date,note,paiement,mois,annee,jour_semaine,nb_achats,total_depense,segment_valeur
0,4018,Handbag,4619.0,2023-02-05,NaN,Credit Card,2023-02,2023,Sunday,15,5805.0,4. VIP
1,4115,Tunic,2456.0,2023-07-11,2.0,Credit Card,2023-07,2023,Tuesday,22,4807.0,4. VIP
2,4019,Tank Top,2102.0,2023-03-23,4.1,Cash,2023-03,2023,Thursday,14,3598.0,4. VIP
3,4097,Leggings,3126.0,2023-03-15,3.2,Cash,2023-03,2023,Wednesday,8,3926.0,4. VIP
4,3997,Wallet,3003.0,2022-11-27,4.7,Cash,2022-11,2022,Sunday,14,3889.0,4. VIP


In [20]:
print("Lignes finales :", len(df_final))
print("Colonnes :", df_final.columns.tolist())
print("\nRépartition par segment :")
print(df_final["segment_valeur"].value_counts())
print("\nMontant total :", df_final["montant"].sum())

Lignes finales : 2750
Colonnes : ['client_id', 'article', 'montant', 'date', 'note', 'paiement', 'mois', 'annee', 'jour_semaine', 'nb_achats', 'total_depense', 'segment_valeur']

Répartition par segment :
segment_valeur
3. Fidèle         794
4. VIP            777
2. Régulier       644
1. Occasionnel    535
Name: count, dtype: int64

Montant total : 430952.0


In [21]:
df_final.to_csv("../data/fashion_looker.csv", index=False)
print("Exporté : data/fashion_looker.csv — prêt pour Looker Studio !")

Exporté : data/fashion_looker.csv — prêt pour Looker Studio !
